In [ ]:





# 2. Imports


import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import torch
import torch.nn as nn

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split

import albumentations as A
import segmentation_models_pytorch as smp


# 3. Paths


# Move one level up from notebook/
BASE_DIR = os.path.dirname(os.getcwd())

DATASET_DIR = os.path.join(
    BASE_DIR,
    "Dataset"
)

PATCH_DIR = "artifacts\\patches"
AUG_DIR = "artifacts\\agumented"

BEST_MODEL_PATH = "models\\best_model.pth"


# 4. TIFF Reader

def read_tiff(path):

    with rasterio.open(path) as src:

        img = src.read()

        img = np.transpose(img, (1,2,0))

        meta = {
            "crs": src.crs,
            "transform": src.transform,
            "height": src.height,
            "width": src.width
        }

    return img, meta


# 5. Process Masks

def process_mask(before_mask, after_mask):

    mask = np.zeros_like(after_mask)

    building = (before_mask == 2)
    damaged = (after_mask == 1)

    mask[building] = 0
    mask[damaged] = 1

    return mask


# 6. Patch Creation

def create_patches(img, mask, size=256, stride=64):

    imgs = []
    masks = []

    H, W = img.shape[:2]

    for i in range(0, H-size, stride):
        for j in range(0, W-size, stride):

            patch_img = img[i:i+size, j:j+size]
            patch_mask = mask[i:i+size, j:j+size]

            if np.sum(patch_mask) > 500:
                keep = True
            else:
                keep = np.random.rand() < 0.2

            if keep:
                imgs.append(patch_img)
                masks.append(patch_mask)

    return imgs, masks


# 7. Augmentation


transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
])
   

    

# 8. Create Patches + Save Augmented Images

os.makedirs(PATCH_DIR, exist_ok=True)
os.makedirs(AUG_DIR, exist_ok=True)

all_samples = []

patch_count = 0
aug_count = 0

for section in os.listdir(DATASET_DIR):

    sec = os.path.join(DATASET_DIR, section)

    if not os.path.isdir(sec):
        continue

    before, _ = read_tiff(os.path.join(sec, "before.tif"))
    after, _ = read_tiff(os.path.join(sec, "after.tif"))

    before_mask, _ = read_tiff(os.path.join(sec, "before_mask.tif"))
    after_mask, _ = read_tiff(os.path.join(sec, "after_mask.tif"))

    before_mask = before_mask[:,:,0]
    after_mask = after_mask[:,:,0]

    h = min(before.shape[0], after.shape[0])
    w = min(before.shape[1], after.shape[1])

    before = cv2.resize(before, (w,h))
    after = cv2.resize(after, (w,h))

    before_mask = cv2.resize(before_mask, (w,h), interpolation=cv2.INTER_NEAREST)
    after_mask = cv2.resize(after_mask, (w,h), interpolation=cv2.INTER_NEAREST)

    mask = process_mask(before_mask, after_mask)

    # Difference channel
    diff = after.astype(np.float32) - before.astype(np.float32)

    combined = np.concatenate([
        before,
        after,
        diff
    ], axis=-1)

    combined = combined.astype(np.float32)

    combined = (
        combined - combined.min()
    ) / (
        combined.max() - combined.min() + 1e-6
    )

    imgs, masks = create_patches(combined, mask)

    for img_patch, mask_patch in zip(imgs, masks):

        all_samples.append((img_patch, mask_patch))

        patch_count += 1

        rgb = (img_patch[:,:,:3] * 255).astype(np.uint8)

        cv2.imwrite(
            f"{PATCH_DIR}/patch_{patch_count}.png",
            rgb
        )

        aug = transform(
            image=img_patch,
            mask=mask_patch
        )

        all_samples.append((aug['image'], aug['mask']))

        aug_img = (aug['image'][:,:,:3] * 255).astype(np.uint8)

        cv2.imwrite(
            f"{AUG_DIR}/aug_{patch_count}.png",
            aug_img
        )

        aug_count += 1

print("Total patches:", patch_count)
print("Total augmented:", aug_count)


# 9. Dataset Class


class DamageDataset(Dataset):

    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img, mask = self.samples[idx]

        img = np.transpose(img, (2,0,1))

        return (
            torch.tensor(img).float(),
            torch.tensor(mask).unsqueeze(0).float()
        )
   

    

# 10. Train/Validation Split

 
dataset = DamageDataset(all_samples)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)
   

    

# 11. Model

 
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=9,
    classes=1
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)
   

    

# 12. Loss Functions

 
def dice_loss(pred, target, smooth=1):

    pred = torch.sigmoid(pred)

    pred = pred.view(-1)
    target = target.view(-1)

    inter = (pred * target).sum()

    return 1 - (
        (2*inter + smooth) /
        (pred.sum() + target.sum() + smooth)
    )
   

 
def loss_fn(pred, target):

    bce = nn.BCEWithLogitsLoss()(pred, target)

    return bce + 2*dice_loss(pred, target)
   

    

# 13. Metrics

 
def calculate_metrics(pred, target, threshold=0.5):

    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    target = target.float()

    pred = pred.view(-1)
    target = target.view(-1)

    TP = (pred * target).sum()
    TN = ((1 - pred) * (1 - target)).sum()
    FP = (pred * (1 - target)).sum()
    FN = ((1 - pred) * target).sum()

    eps = 1e-6

    accuracy = (TP + TN) / (TP + TN + FP + FN + eps)

    precision = TP / (TP + FP + eps)

    recall = TP / (TP + FN + eps)

    f1 = 2 * precision * recall / (precision + recall + eps)

    iou = TP / (TP + FP + FN + eps)

    return {
        "Accuracy": accuracy.item(),
        "Precision": precision.item(),
        "Recall": recall.item(),
        "F1 Score": f1.item(),
        "IoU": iou.item()
    }
   

    

# 14. Optimizer + Scheduler

 
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5
)
   

    

# 15. Training Loop

 
best_iou = 0

train_losses = []
val_ious = []

EPOCHS = 75
   

 
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for imgs, masks in tqdm(train_loader):

        imgs = imgs.to(device)
        masks = masks.to(device)

        preds = model(imgs)

        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    train_losses.append(avg_loss)

    # Validation
    model.eval()

    total_metrics = {
        "Accuracy":0,
        "Precision":0,
        "Recall":0,
        "F1 Score":0,
        "IoU":0
    }

    with torch.no_grad():

        for imgs, masks in val_loader:

            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)

            metrics = calculate_metrics(preds, masks)

            for k in total_metrics:
                total_metrics[k] += metrics[k]

    for k in total_metrics:
        total_metrics[k] /= len(val_loader)

    avg_iou = total_metrics["IoU"]

    scheduler.step(avg_iou)

    val_ious.append(avg_iou)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {avg_loss:.4f}")

    for k, v in total_metrics.items():
        print(f"{k}: {v:.4f}")

    if avg_iou > best_iou:

        best_iou = avg_iou

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("✅ Best model saved")
   

    

# 16. Training Curves

 
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses)
plt.title("Training Loss")

plt.subplot(1,2,2)
plt.plot(val_ious)
plt.title("Validation IoU")

plt.show()
   

    

# 17. Load Best Model

 
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()
   

    

# 18. Evaluate Best Model

 
total_metrics = {
    "Accuracy":0,
    "Precision":0,
    "Recall":0,
    "F1 Score":0,
    "IoU":0
}

with torch.no_grad():

    for imgs, masks in val_loader:

        imgs = imgs.to(device)
        masks = masks.to(device)

        preds = model(imgs)

        metrics = calculate_metrics(preds, masks)

        for k in total_metrics:
            total_metrics[k] += metrics[k]

for k in total_metrics:
    total_metrics[k] /= len(val_loader)

print("\n===== BEST MODEL PERFORMANCE =====\n")

for k, v in total_metrics.items():
    print(f"{k}: {v:.4f}")
   

    

# 19. TIFF Prediction Function

 
def predict_tiff(model, before_path, after_path):

    before, _ = read_tiff(before_path)
    after, _ = read_tiff(after_path)

    h = min(before.shape[0], after.shape[0])
    w = min(before.shape[1], after.shape[1])

    before = cv2.resize(before, (w,h))
    after = cv2.resize(after, (w,h))

    H, W = before.shape[:2]

    pred_map = np.zeros((H,W))
    count_map = np.zeros((H,W))

    for i in range(0, H-256, 64):
        for j in range(0, W-256, 64):

            b = before[i:i+256, j:j+256]
            a = after[i:i+256, j:j+256]

            diff = a.astype(np.float32) - b.astype(np.float32)

            combined = np.concatenate([
                b,
                a,
                diff
            ], axis=-1)

            combined = combined.astype(np.float32)

            combined = (
                combined - combined.min()
            ) / (
                combined.max() - combined.min() + 1e-6
            )

            combined = np.transpose(combined, (2,0,1))

            inp = torch.tensor(combined).unsqueeze(0).to(device)

            with torch.no_grad():
                pred = torch.sigmoid(model(inp)).cpu().numpy()[0,0]

            pred_map[i:i+256, j:j+256] += pred
            count_map[i:i+256, j:j+256] += 1

    pred_map = pred_map / (count_map + 1e-6)

    return pred_map
   

    



In [ ]:

# 20. TIFF Testing

 
BEFORE_TEST = "Dataset\section_0\\after_mask.tif"
AFTER_TEST  = "Dataset\section_0\\before_mask.tif"
   

 
before, _ = read_tiff(BEFORE_TEST)
after, _  = read_tiff(AFTER_TEST)

before_rgb = before[:,:,:3].astype(np.uint8)
after_rgb  = after[:,:,:3].astype(np.uint8)

h = min(before_rgb.shape[0], after_rgb.shape[0])
w = min(before_rgb.shape[1], after_rgb.shape[1])

before_rgb = cv2.resize(before_rgb, (w,h))
after_rgb  = cv2.resize(after_rgb, (w,h))

pred = predict_tiff(model, BEFORE_TEST, AFTER_TEST)

pred = cv2.resize(pred, (w,h))

pred_norm = pred / (pred.max() + 1e-6)

# boost visibility
pred_norm = np.power(pred_norm, 0.6)

print("Prediction Min:", pred_norm.min())
print("Prediction Max:", pred_norm.max())
   

    

# 21. Damage Mask + Percentage

 
threshold = 0.25

damage_mask = (pred_norm > threshold).astype(np.uint8)

num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
    damage_mask,
    connectivity=8
)

clean_mask = np.zeros_like(damage_mask)

for i in range(1, num_labels):

    area = stats[i, cv2.CC_STAT_AREA]

    if area > 40:
        clean_mask[labels == i] = 1

damage_mask = clean_mask

damage_pixels = np.sum(damage_mask)

# urban/building-like regions
urban_mask = diff_gray > 25

urban_pixels = np.sum(urban_mask)

damage_percent = (
    damage_pixels / (urban_pixels + 1e-6)
) * 100

print(f"Damage %: {damage_percent:.2f}%")
   

    

# 22. Visualization

 
diff = cv2.absdiff(before_rgb, after_rgb)

diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
   

 
heatmap = cv2.applyColorMap(
    (pred_norm * 255).astype(np.uint8),
    cv2.COLORMAP_HOT
)

heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

heatmap = cv2.resize(heatmap, (w,h))
   

 
overlay = cv2.addWeighted(
    after_rgb,
    0.45,
    heatmap,
    0.55,
    0
)
   

 
highlight = after_rgb.copy()

highlight[damage_mask == 1] = [255,0,0]

damage_overlay = cv2.addWeighted(
    after_rgb,
    0.7,
    highlight,
    0.3,
    0
)
   

 
plt.figure(figsize=(22,12))

plt.subplot(2,3,1)
plt.imshow(before_rgb)
plt.title("Before Image")
plt.axis('off')

plt.subplot(2,3,2)
plt.imshow(after_rgb)
plt.title("After Image")
plt.axis('off')

plt.subplot(2,3,3)
plt.imshow(diff_gray, cmap='gray')
plt.title("Absolute Difference")
plt.axis('off')

plt.subplot(2,3,4)
plt.imshow(pred_norm, cmap='hot')
plt.title("Prediction Heatmap")
plt.axis('off')

plt.subplot(2,3,5)
plt.imshow(overlay)
plt.title("Prediction Overlay")
plt.axis('off')

plt.subplot(2,3,6)
plt.imshow(damage_overlay)
plt.title(f"Damage Highlighted Damage % = {damage_percent:.2f}%")
plt.axis('off')

plt.tight_layout()
plt.show()
   

    

# 23. Save Outputs

 
cv2.imwrite(
    "/content/damage_overlay.png",
    cv2.cvtColor(damage_overlay, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    "/content/damage_mask.png",
    damage_mask * 255
)

print("Saved outputs successfully")
   